# TOST drop-in: formal equivalence / non-inferiority for the CIFAR-10 median-vs-Mahalanobis tie



In [ ]:
# ===================== [PREAMBLE] (skip if reusing your kernel) =====================
import os, io as _io, subprocess, pickle, json
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.models as models
from torchvision.transforms.functional import gaussian_blur
from PIL import Image as _Image
from sklearn.metrics import roc_auc_score
from sklearn.covariance import LedoitWolf

SEED=42; device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
np.random.seed(SEED); torch.manual_seed(SEED)

def _find(name, maxdepth=8):
    roots=['/home','/root','/workspace',os.path.expanduser('~'),'.','..','../..','.']
    res=[]
    for r in roots:
        if not os.path.exists(r): continue
        try:
            out=subprocess.run(['find',r,'-maxdepth',str(maxdepth),'-type','f','-name',name],
                               capture_output=True,text=True,timeout=20).stdout.strip()
            if out: res+=[p for p in out.split('\n') if p]
        except: pass
    return sorted(set(res))

CIFAR_MEAN=[0.4914,0.4822,0.4465]; CIFAR_STD=[0.2470,0.2435,0.2616]
def make_pp(ds):
    mean=torch.tensor(CIFAR_MEAN).view(1,3,1,1).to(device); std=torch.tensor(CIFAR_STD).view(1,3,1,1).to(device)
    return lambda x:(x/255.0-mean)/std
def load_backbone(ds):
    ck=(_find('resnet50_cifar10_finetuned.pt') or [None])[0]
    m=models.resnet50(weights=None); m.fc=nn.Linear(2048,10)
    m.load_state_dict(torch.load(ck,map_location=device)['state_dict']); return m.to(device).eval()
def gb(x,s):
    k=int(2*np.ceil(3*s)+1); k=k+1 if k%2==0 else k
    return gaussian_blur(x,kernel_size=k,sigma=s)
def to224(img):
    if img.dim()==3: img=img.unsqueeze(0)
    img=img.float()
    if img.shape[-1]!=224: img=F.interpolate(img,size=(224,224),mode='bicubic',align_corners=False)
    return img.clamp(0,255)
def jpeg(x,q=75):
    a=x.detach().squeeze(0).permute(1,2,0).clamp(0,255).byte().cpu().numpy()
    b=_io.BytesIO(); _Image.fromarray(a).save(b,format='JPEG',quality=int(q)); b.seek(0)
    return torch.from_numpy(np.array(_Image.open(b).convert('RGB'))).float().permute(2,0,1)
def jpeg_batch(b,q=75): return torch.stack([jpeg(b[i:i+1]) for i in range(b.shape[0])]).to(b.device)
def median3(x):
    p=F.pad(x,(1,1,1,1),mode='reflect')
    return p.unfold(2,3,1).unfold(3,3,1).contiguous().view(*x.shape,9).median(-1).values
def feat_hfe(b): return ((b-gb(b,0.5)).abs().flatten(1).mean(1)/255.0).detach().cpu().numpy()
def feat_gl(b,bb,pp,glsig):
    with torch.no_grad():
        p0=F.softmax(bb(pp(b)),1); p1=F.softmax(bb(pp(gb(b,glsig))),1)
    return (p0-p1).abs().sum(1).cpu().numpy()
def feat_predl1(b,bb,pp):
    with torch.no_grad():
        p0=F.softmax(bb(pp(b)),1); sq=median3(jpeg_batch(b)).clamp(0,255); p2=F.softmax(bb(pp(sq)),1)
    return (p0-p2).abs().sum(1).cpu().numpy()

# 4-layer deep features for Mahalanobis (global-average-pooled layer1..layer4)
class Feats:
    def __init__(self, model):
        self.model=model; self.f={}
        self.h=[model.layer1.register_forward_hook(self._mk('l1')),
                model.layer2.register_forward_hook(self._mk('l2')),
                model.layer3.register_forward_hook(self._mk('l3')),
                model.layer4.register_forward_hook(self._mk('l4'))]
    def _mk(self,n):
        def hook(m,i,o): self.f[n]=o.mean((2,3)).detach()
        return hook
    def __call__(self,x):
        self.f={}
        with torch.no_grad(): self.model(x)
        return [self.f['l1'],self.f['l2'],self.f['l3'],self.f['l4']]

def extract_deep(imgs, fe, pp, bs=64):
    L=[[],[],[],[]]
    for i in range(0,len(imgs),bs):
        b=imgs[i:i+bs].to(device); fl=fe(pp(b))
        for k in range(4): L[k].append(fl[k].cpu().numpy())
    return [np.concatenate(x,0) for x in L]

def fit_maha_cc(Fl, labels):
    labels=np.asarray(labels); cls=np.unique(labels); assert len(cls)>1, 'calibration has <=1 class; predicted-label assignment should give ~10 on CIFAR-10'
    Ws=[]; WMs=[]
    for f in Fl:
        mu=np.stack([f[labels==c].mean(0) for c in cls])
        cen=np.concatenate([f[labels==c]-mu[i] for i,c in enumerate(cls)],0)
        cov=LedoitWolf().fit(cen).covariance_; L=np.linalg.cholesky(np.linalg.inv(cov))
        Ws.append(L); WMs.append(mu.dot(L))
    return Ws, WMs, cls
def maha(Fl, Ws, WMs):
    s=0
    for f,L,WM in zip(Fl,Ws,WMs):
        w=f.dot(L); d2=(w**2).sum(1)[:,None]+(WM**2).sum(1)[None,:]-2.0*w.dot(WM.T)
        s=s+(-d2.min(1))
    return s   # high = clean ; anomaly uses -maha

def find_mixed():
    out={}
    for p in _find('mixed_dataset.pkl'):
        pl=p.lower()
        if 'cifar10' in pl or 'cifar_10' in pl: out.setdefault('CIFAR-10',p)
    return out
print('[PREAMBLE] ready')


In [ ]:
# ===================== build scores + paired bootstrap + TOST =====================
SEED=42; glsig=0.5; B=5000; MARGINS=[0.02,0.03,0.05]
def half(n, seed=SEED):
    rng=np.random.RandomState(seed); idx=np.arange(n); rng.shuffle(idx); return idx[:n//2], idx[n//2:]

bb=load_backbone('CIFAR-10'); pp=make_pp('CIFAR-10'); fe=Feats(bb)
mixed=pickle.load(open(find_mixed()['CIFAR-10'],'rb'))
clean=[(to224(im).cpu(), int(lb)) for (im,lb,atk) in mixed if atk=='clean']
adv  =[to224(im).cpu() for (im,lb,atk) in mixed if atk!='clean']
rng=np.random.RandomState(SEED); ci_perm=rng.permutation(len(clean))[:500]
clean=[clean[i] for i in ci_perm]
Xc=torch.cat([c[0] for c in clean],0); yc=np.array([c[1] for c in clean]); Xa=torch.cat(adv,0)
ci,ti=half(len(clean))
Xcal=Xc[ci]; Xte=Xc[ti]
# CIFAR-10 cache labels can be degenerate (single value); assign class = model prediction.
# The classifier is highly accurate on clean CIFAR-10, so predicted labels approximate GT and give a
# genuine 10-class class-conditional structure (same approach used for the ImageNet baseline).
ycal=[]
for i in range(0,len(Xcal),64):
    b=Xcal[i:i+64].to(device)
    with torch.no_grad(): ycal.append(bb(pp(b)).argmax(1).cpu().numpy())
ycal=np.concatenate(ycal)
# hard negatives on clean test half: matched noise + JPEG (as in Table VI)
noise=(Xte+torch.randn_like(Xte)*8.0).clamp(0,255); jp=jpeg_batch(Xte.to(device)).cpu()
Xhn=torch.cat([noise,jp],0)
ai_c,ai_t=half(len(adv)); hi_c,hi_t=half(Xhn.shape[0])

# ---- median aggregator anomaly ----
def scal(X):
    H=[];G=[];P=[]
    for i in range(0,len(X),64):
        b=X[i:i+64].to(device); H.append(feat_hfe(b)); G.append(feat_gl(b,bb,pp,glsig)); P.append(feat_predl1(b,bb,pp))
    return np.concatenate(H),np.concatenate(G),np.concatenate(P)
Hc,Gc,Pc=scal(Xcal); muH,sdH=Hc.mean(),Hc.std()+1e-8; muG,sdG=Gc.mean(),Gc.std()+1e-8; muP,sdP=Pc.mean(),Pc.std()+1e-8
def med_anom(X):
    H,G,P=scal(X); Z=np.stack([(H-muH)/sdH,(G-muG)/sdG,(P-muP)/sdP],1); return np.median(Z,1)
muS=np.median(np.stack([(Hc-muH)/sdH,(Gc-muG)/sdG,(Pc-muP)/sdP],1),1).mean()
med_te=np.abs(med_anom(Xte)-muS); med_hn=np.abs(med_anom(Xhn)-muS); med_adv=np.abs(med_anom(Xa)-muS)

# ---- class-conditional Mahalanobis anomaly ----
Fcal=extract_deep(Xcal, fe, pp); Ws,WMs,cls=fit_maha_cc(Fcal, ycal)
print('Maha calib classes:', len(cls))
Fte=extract_deep(Xte, fe, pp); Fhn=extract_deep(Xhn, fe, pp); Fa=extract_deep(Xa, fe, pp)
mah_te=-maha(Fte,Ws,WMs); mah_hn=-maha(Fhn,Ws,WMs); mah_adv=-maha(Fa,Ws,WMs)

# ---- leakage-safe TEST set: neg = clean_te ∪ hn[hi_t], pos = adv[ai_t] ----
med_neg=np.concatenate([med_te, med_hn[hi_t]]); med_pos=med_adv[ai_t]
mah_neg=np.concatenate([mah_te, mah_hn[hi_t]]); mah_pos=mah_adv[ai_t]
def auroc(neg,pos): return roc_auc_score(np.r_[np.zeros(len(neg)),np.ones(len(pos))], np.r_[neg,pos])
au_med=auroc(med_neg,med_pos); au_mah=auroc(mah_neg,mah_pos)
print(f'median AUROC={au_med:.4f}  Mahalanobis AUROC={au_mah:.4f}  observed diff={au_med-au_mah:+.4f}')

# ---- paired bootstrap: same resampled indices for both detectors ----
rng=np.random.RandomState(SEED); nN=len(med_neg); nP=len(med_pos); D=np.empty(B)
for b in range(B):
    ni=rng.randint(0,nN,nN); pi=rng.randint(0,nP,nP)
    y=np.r_[np.zeros(nN),np.ones(nP)]
    mb=roc_auc_score(y, np.r_[med_neg[ni],med_pos[pi]])
    hb=roc_auc_score(y, np.r_[mah_neg[ni],mah_pos[pi]])
    D[b]=mb-hb
p2_5,p5,p95,p97_5=np.percentile(D,[2.5,5,95,97.5])
point=au_med-au_mah
ci90=[float(p5),float(p95)]; ci95=[float(p2_5),float(p97_5)]
dstar_equiv=float(max(abs(p5),abs(p95))); dstar_noninf=float(max(0.0,-p5))

per_margin={}
for delta in MARGINS:
    per_margin[f'{delta}']={'equivalence': bool(p5>-delta and p95<delta),
                            'non_inferiority': bool(p5>-delta)}

res={'cifar10':{'auroc_median':round(au_med,4),'auroc_maha':round(au_mah,4),
                'observed_diff':round(point,4),'ci90':[round(x,4) for x in ci90],
                'ci95':[round(x,4) for x in ci95],
                'dstar_equivalence':round(dstar_equiv,4),'dstar_noninferiority':round(dstar_noninf,4),
                'margins':per_margin,'B':B,'n_neg':int(nN),'n_pos':int(nP),'maha_classes':int(len(cls))}}
OUT='./tost_results'; os.makedirs(OUT,exist_ok=True)
json.dump(res, open(os.path.join(OUT,'tost_cifar10.json'),'w'), indent=2)

print('\n=== TOST / non-inferiority (CIFAR-10, median - Mahalanobis) ===')
print(f'observed diff {point:+.4f} | 90% CI [{p5:+.4f}, {p95:+.4f}] | 95% CI [{p2_5:+.4f}, {p97_5:+.4f}]')
print(f'smallest supportable margins: equivalence delta* = {dstar_equiv:.4f} | non-inferiority delta* = {dstar_noninf:.4f}')
for delta in MARGINS:
    m=per_margin[f'{delta}']
    print(f'  margin {delta:.2f}: equivalence={"HOLDS" if m["equivalence"] else "FAILS"} | non-inferiority={"HOLDS" if m["non_inferiority"] else "FAILS"}')
print('\nsaved', os.path.join(OUT,'tost_cifar10.json'))
